# EDA — ChurnGuard : Dataset Rivalytics

**5 tables** : accounts, subscriptions, feature_usage, churn_events, support_tickets  
**Target** : `churn_flag` dans `rivalytics_accounts`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

matplotlib.rcParams['figure.figsize'] = (10, 4)
DATA_DIR = Path('../data')

accounts       = pd.read_csv(DATA_DIR / 'rivalytics_accounts.csv')
subscriptions  = pd.read_csv(DATA_DIR / 'rivalytics_subscriptions.csv')
feature_usage  = pd.read_csv(DATA_DIR / 'rivalytics_feature_usage.csv')
churn_events   = pd.read_csv(DATA_DIR / 'rivalytics_churn_events.csv')
support_tickets = pd.read_csv(DATA_DIR / 'rivalytics_support_tickets.csv')

print('accounts      :', accounts.shape)
print('subscriptions :', subscriptions.shape)
print('feature_usage :', feature_usage.shape)
print('churn_events  :', churn_events.shape)
print('support_tickets:', support_tickets.shape)

---
## 1. Distribution de la target

In [ ]:
churn_counts = accounts['churn_flag'].value_counts()
churn_pct    = accounts['churn_flag'].value_counts(normalize=True).round(3) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
churn_counts.plot.bar(ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Comptes churned vs actifs (n)')
axes[0].set_xticklabels(['Non-churn', 'Churn'], rotation=0)

churn_pct.plot.pie(ax=axes[1], labels=['Non-churn (78%)', 'Churn (22%)'],
                   colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Déséquilibre des classes')
plt.tight_layout()
plt.show()

print('Churn : 22% — déséquilibre modéré → class_weight="balanced" suffira probablement (SMOTE si F1 < 0.70)')

---
## 2. Table `rivalytics_accounts`

In [ ]:
print('Shape :', accounts.shape)
print('Nulls :')
print(accounts.isnull().sum())
accounts.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Churn par plan_tier
churn_by_plan = accounts.groupby('plan_tier')['churn_flag'].mean().sort_values(ascending=False)
churn_by_plan.plot.bar(ax=axes[0], color='#e74c3c')
axes[0].set_title('Taux de churn par plan_tier')
axes[0].set_ylabel('Taux churn')
axes[0].set_xticklabels(churn_by_plan.index, rotation=0)

# Churn par industry
churn_by_industry = accounts.groupby('industry')['churn_flag'].mean().sort_values(ascending=False)
churn_by_industry.plot.bar(ax=axes[1], color='#3498db')
axes[1].set_title('Taux de churn par industrie')
axes[1].set_xticklabels(churn_by_industry.index, rotation=30, ha='right')

# Churn par referral_source
churn_by_ref = accounts.groupby('referral_source')['churn_flag'].mean().sort_values(ascending=False)
churn_by_ref.plot.bar(ax=axes[2], color='#9b59b6')
axes[2].set_title('Taux de churn par referral_source')
axes[2].set_xticklabels(churn_by_ref.index, rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Tenure : durée depuis signup
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
reference_date = pd.Timestamp('2025-01-01')  # date de référence dataset
accounts['tenure_days'] = (reference_date - accounts['signup_date']).dt.days

fig, ax = plt.subplots(figsize=(10, 4))
for churn_val, label, color in [(False, 'Non-churn', '#2ecc71'), (True, 'Churn', '#e74c3c')]:
    accounts[accounts['churn_flag'] == churn_val]['tenure_days'].hist(
        bins=30, ax=ax, alpha=0.6, label=label, color=color
    )
ax.set_title('Distribution tenure_days par statut churn')
ax.set_xlabel('Jours depuis signup')
ax.legend()
plt.tight_layout()
plt.show()

print('Médiane tenure (non-churn):', accounts[~accounts['churn_flag']]['tenure_days'].median())
print('Médiane tenure (churn):', accounts[accounts['churn_flag']]['tenure_days'].median())

---
## 3. Table `rivalytics_subscriptions`

In [ ]:
print('Shape :', subscriptions.shape)
print('Nulls :')
print(subscriptions.isnull().sum())
subscriptions.head()

In [ ]:
print('Abonnements actifs (end_date null):', subscriptions['end_date'].isna().sum())
print('Trials (mrr_amount=0):', (subscriptions['mrr_amount'] == 0).sum())
print('Avec upgrade :', subscriptions['upgrade_flag'].sum())
print('Avec downgrade :', subscriptions['downgrade_flag'].sum())

# Distribution MRR (hors trials)
sub_paid = subscriptions[subscriptions['mrr_amount'] > 0]
fig, ax = plt.subplots(figsize=(10, 4))
for churn_val, label, color in [(False, 'Non-churn', '#2ecc71'), (True, 'Churn', '#e74c3c')]:
    sub_paid[sub_paid['churn_flag'] == churn_val]['mrr_amount'].hist(
        bins=40, ax=ax, alpha=0.6, label=label, color=color
    )
ax.set_title('Distribution MRR par statut churn (hors trials)')
ax.set_xlabel('MRR (USD)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Table `rivalytics_feature_usage`

In [ ]:
print('Shape :', feature_usage.shape)
print('Features uniques :', feature_usage['feature_name'].nunique())
print('Nulls :')
print(feature_usage.isnull().sum())
print()
print('error_count > 0 :', (feature_usage['error_count'] > 0).sum(), '/', len(feature_usage))
print('beta sessions :', feature_usage['is_beta_feature'].sum(), '/', len(feature_usage))
feature_usage.head()

In [ ]:
# Agréger par subscription pour joindre avec churn_flag
fu_with_churn = feature_usage.merge(
    subscriptions[['subscription_id', 'account_id', 'churn_flag']],
    on='subscription_id', how='left'
).merge(accounts[['account_id', 'churn_flag']].rename(columns={'churn_flag': 'account_churn'}),
        on='account_id', how='left')

# error_rate vs churn (agrégé par account)
usage_per_account = fu_with_churn.groupby(['account_id', 'account_churn']).agg(
    total_errors=('error_count', 'sum'),
    total_sessions=('usage_count', 'sum')
).reset_index()
usage_per_account['error_rate'] = usage_per_account['total_errors'] / usage_per_account['total_sessions'].replace(0, 1)

print('Médiane error_rate (non-churn):', usage_per_account[~usage_per_account['account_churn']]['error_rate'].median().round(4))
print('Médiane error_rate (churn):', usage_per_account[usage_per_account['account_churn']]['error_rate'].median().round(4))

fig, ax = plt.subplots(figsize=(10, 4))
for churn_val, label, color in [(False, 'Non-churn', '#2ecc71'), (True, 'Churn', '#e74c3c')]:
    usage_per_account[usage_per_account['account_churn'] == churn_val]['error_rate'].hist(
        bins=30, ax=ax, alpha=0.6, label=label, color=color
    )
ax.set_title('Distribution error_rate par statut churn')
ax.set_xlabel('Taux d erreur (erreurs/sessions)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Table `rivalytics_churn_events`

In [ ]:
print('Shape :', churn_events.shape)
print('is_reactivation:', churn_events['is_reactivation'].value_counts().to_dict())
churn_events.head()

In [ ]:
# Filtrage réactivations (double-comptage)
ce_clean = churn_events[churn_events['is_reactivation'] == False]
print('Événements réels de churn (hors réactivation) :', len(ce_clean))
print('Comptes uniques avec churn event :', ce_clean['account_id'].nunique())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ce_clean['reason_code'].value_counts().plot.bar(ax=axes[0], color='#e74c3c')
axes[0].set_title('Distribution reason_code (hors réactivation)')
axes[0].set_xticklabels(ce_clean['reason_code'].value_counts().index, rotation=30, ha='right')

ce_clean['refund_amount_usd'].hist(bins=30, ax=axes[1], color='#e67e22')
axes[1].set_title('Distribution remboursements (USD)')
axes[1].set_xlabel('Montant remboursé')

plt.tight_layout()
plt.show()

print('preceding_downgrade avant churn :', ce_clean['preceding_downgrade_flag'].sum(), '/', len(ce_clean))

---
## 6. Table `rivalytics_support_tickets`

In [ ]:
print('Shape :', support_tickets.shape)
print('Nulls :')
print(support_tickets.isnull().sum())
support_tickets.head()

In [ ]:
# Joindre avec churn_flag
st_with_churn = support_tickets.merge(accounts[['account_id', 'churn_flag']], on='account_id', how='left')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# satisfaction_score
for churn_val, label, color in [(False, 'Non-churn', '#2ecc71'), (True, 'Churn', '#e74c3c')]:
    st_with_churn[st_with_churn['churn_flag'] == churn_val]['satisfaction_score'].dropna().hist(
        bins=20, ax=axes[0], alpha=0.6, label=label, color=color
    )
axes[0].set_title('satisfaction_score par statut churn')
axes[0].legend()

# resolution_time_hours
for churn_val, label, color in [(False, 'Non-churn', '#2ecc71'), (True, 'Churn', '#e74c3c')]:
    st_with_churn[st_with_churn['churn_flag'] == churn_val]['resolution_time_hours'].hist(
        bins=30, ax=axes[1], alpha=0.6, label=label, color=color
    )
axes[1].set_title('Temps de résolution par statut churn')
axes[1].legend()

# escalation_flag par churn
esc_by_churn = st_with_churn.groupby('churn_flag')['escalation_flag'].mean()
esc_by_churn.plot.bar(ax=axes[2], color=['#2ecc71', '#e74c3c'])
axes[2].set_title('Taux d escalation par statut churn')
axes[2].set_xticklabels(['Non-churn', 'Churn'], rotation=0)
axes[2].set_ylabel('Taux escalation')

plt.tight_layout()
plt.show()

print('Nulls satisfaction_score :', support_tickets['satisfaction_score'].isna().sum(), '→ imputation par médiane')
print('Médiane satisfaction (non-churn):', st_with_churn[~st_with_churn['churn_flag']]['satisfaction_score'].median())
print('Médiane satisfaction (churn):', st_with_churn[st_with_churn['churn_flag']]['satisfaction_score'].median())

---
## 7. Observations clés

In [ ]:
# Matrice de corrélation sur le dataset mergé
from src.preprocessing.loader import load_all
from src.preprocessing.cleaner import clean_accounts, clean_subscriptions, clean_feature_usage, clean_churn_events, clean_support_tickets
from src.preprocessing.merger import merge_all
import sys
sys.path.insert(0, '..')

raw = load_all()
merged = merge_all(
    clean_accounts(raw['accounts']),
    clean_subscriptions(raw['subscriptions']),
    clean_feature_usage(raw['feature_usage']),
    clean_churn_events(raw['churn_events']),
    clean_support_tickets(raw['support_tickets']),
)

num_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
corr_with_churn = merged[num_cols + ['churn_flag']].corr()['churn_flag'].drop('churn_flag').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
corr_with_churn.head(15).plot.barh(ax=ax, color=['#e74c3c' if v > 0 else '#2ecc71' for v in corr_with_churn.head(15)])
ax.set_title('Top 15 features corrélées avec churn_flag')
ax.set_xlabel('Corrélation de Pearson')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print(corr_with_churn.head(15).round(3).to_string())

---
## Synthèse — Observations pour le feature engineering

| Observation | Impact attendu |
|---|---|
| Déséquilibre classes : 78%/22% | `class_weight='balanced'` sur les modèles |
| `satisfaction_score` : 825 nulls (41%) | Imputation médiane dans cleaner |
| `is_reactivation=True` : 61 events à filtrer | Déjà géré dans cleaner |
| `feature_usage` joint via `subscription_id` | Nécessite l'étape de résolution account_id |
| `churn_event_count` / `preceding_downgrade_flag` | Features fortes (mais vérifier fuite de données) |
| `escalation_count` corrélé positivement au churn | Feature de support à inclure |
| 8 comptes sans ticket → nulls dans support_agg | Imputer à 0 pour ticket_count, escalation_count |

**Prochain jalon** : Phase 2 — feature engineering (`src/preprocessing/features.py`)